# Pre-processing for augmented data generated with Qwen-VL 

## Preperation

### Import necessary libraries

In [1]:
import numpy as np
import pandas as pd
import os
import json
import ast

### Load `.csv` dataset 

In [2]:
DIR_DATASETS = './data/quen-vl'

raw_datas = list()

for root, dirs, files in os.walk(DIR_DATASETS):
    files.sort()
    for file in files:
        filepath = os.path.join(root, file)
        try:
            if file.endswith(".csv"):
                raw_datas.append(pd.read_csv(filepath))

                print(f"Complete reading {file}")

        except Exception as e:
            print(f"Error reading {filepath}: {e}")

Complete reading res1.csv
Complete reading res2.csv
Complete reading res3.csv
Complete reading res4.csv


### Merge data files into a big dataset

Check if the raw data files has the same amount of columns

In [3]:
for i in range(len(raw_datas) - 1):
    if raw_datas[i].shape[0] != raw_datas[i + 1].shape[0]:
        print("Data files has different amount of rows!")
        break
else:
    print(f"All data files has the same amount of rows: {raw_datas[0].shape[0]}")

All data files has the same amount of rows: 6000


We merged all raw data files together into one big `Dataframe`

In [4]:
raw_dataset = pd.concat(raw_datas, axis=1)

display(raw_dataset.head())
print(raw_dataset.shape)

,img_clarityScore,img_clarityReason,img_occlusionScore,img_occlusionReason,img_diff_abilityScore,img_diff_abilityReason,img_object_densityScore,img_object_densityReason,img_interaction_levelScore,img_interaction_levelReason,...,answer_to_imagepercentage,answer_to_imagematch_count,question_to_answercount,question_to_answermatch_count,question_to_answerpercentage,answer_to_imagefinal_rule_application,answer_to_imagecount_unsupported,answer_to_imagecount_total,question_to_answercount_mismatch,question_to_answercount_total
0,7,The image has moderate clarity with some noise...,9,The main subjects are fully visible without si...,9,"Objects like the cake, candles, and people are...",8,"There are several objects including the cake, ...",4,There is a mild interaction where the woman ap...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7,The image has moderate clarity with some noise...,8,"The main subject, the traffic lights, are most...",8,"Objects like traffic lights, street signs, and...",6,There are several objects including traffic li...,2,There are no clear interactions between object...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,The image has moderate clarity with some noise...,9,"The main subject, the toilet, is fully visible...",9,The objects are clearly distinguishable by the...,5,There are five distinct objects in the image: ...,2,There are no clear interactions between the ob...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7,The image has moderate clarity with some noise...,8,"The puppy is partially occluded by the shoe, b...",9,The puppy and the shoe are easily distinguisha...,4,There are only two main objects in the image: ...,2,There is no clear interaction between the obje...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7,The image has good clarity with clear details ...,9,"The main subject, the living room, is fully vi...",9,"Objects like furniture, appliances, and decor ...",8,There are around 10-15 distinct objects visibl...,2,There are no clear interactions between object...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


(6000, 74)


In [6]:
print(raw_dataset.isnull().sum())

img_clarityScore                            0
img_clarityReason                           0
img_occlusionScore                          0
img_occlusionReason                         0
img_diff_abilityScore                       0
                                         ... 
answer_to_imagefinal_rule_application    5998
answer_to_imagecount_unsupported         5998
answer_to_imagecount_total               5998
question_to_answercount_mismatch         5999
question_to_answercount_total            5999
Length: 74, dtype: int64


In [ ]:
rename_mapping = {
    # EIP (Evaluate Image Prompt)
    'img_clarityScore': 'eip.img_clarity.Score',
    'img_clarityReason': 'eip.img_clarity.Reason',
    'img_occlusionScore': 'eip.img_occlusion.Score',
    'img_occlusionReason': 'eip.img_occlusion.Reason',
    'img_diff_abilityScore': 'eip.img_diff_ability.Score',
    'img_diff_abilityReason': 'eip.img_diff_ability.Reason',
    'img_object_densityScore': 'eip.img_object_density.Score',
    'img_interaction_levelScore': 'eip.img_interaction_level.Score',
    'img_interaction_levelReason': 'eip.img_interaction_level.Reason',
    'img_scene_clutterScore': 'eip.img_scene_clutter.Score',

    # IDP (Image Diversity Prompt)
    'Img_scene_type': 'idp.Img_scene_type',
    'Img_main_object': 'idp.Img_main_object',
    'Image_mainobj_descrip': 'idp.Image_mainobj_descrip',
    'Cultural_context': 'idp.Cultural_context',
    'Demographic_signals': 'idp.Demographic_signals',
    'Scene_typicality_score': 'idp.Scene_typicality_score',

    # ETP (Evaluate Text Prompt)
    'txt_grammarScore_for_question': 'etp.txt_grammar.Score_for_question',
    'txt_grammarReason_for_question': 'etp.txt_grammar.Reason_for_question',
    'txt_grammarScore_for_answers': 'etp.txt_grammar.Score_for_answers',
    'txt_grammarReason_for_answers': 'etp.txt_grammar.Reason_for_answers',
    'txt_unambiguityScore_for_question': 'etp.txt_unambiguity.Score_for_question',
    'txt_unambiguityReason_for_question': 'etp.txt_unambiguity.Reason_for_question',
    'txt_unambiguityScore_for_answers': 'etp.txt_unambiguity.Score_for_answers',
    'txt_unambiguityReason_for_answers': 'etp.txt_unambiguity.Reason_for_answers',
    'txt_qa_structureScore_for_question': 'etp.txt_qa_structure.Score_for_question',
    'txt_qa_structureReason_for_question': 'etp.txt_qa_structure.Reason_for_question',
    'txt_qa_structureScore_for_answers': 'etp.txt_qa_structure.Score_for_answers',
    'txt_qa_structureReason_for_answers': 'etp.txt_qa_structure.Reason_for_answers',
    'syntactic_complexityScore_for_question': 'etp.syntactic_complexity.Score_for_question',
    'syntactic_complexityReason_for_question': 'etp.syntactic_complexity.Reason_for_question',
    'syntactic_complexityScore_for_answers': 'etp.syntactic_complexity.Score_for_answers',
    'syntactic_complexityReason_for_answers': 'etp.syntactic_complexity.Reason_for_answers',
    'language_naturalnessScore_for_question': 'etp.language_naturalness.Score_for_question',
    'language_naturalnessReason_for_question': 'etp.language_naturalness.Reason_for_question',
    'language_naturalnessScore_for_answers': 'etp.language_naturalness.Score_for_answers',
    'language_naturalnessReason_for_answers': 'etp.language_naturalness.Reason_for_answers',

    # VQAC (Visual Question Answers Correlation Prompt)
    'question_to_imageresponse': 'vqac.question_to_image.response',
    'question_to_imagereason': 'vqac.question_to_image.reason',
    'answer_to_imageresponse': 'vqac.answer_to_image.response',
    'answer_to_imageoverall_response': 'vqac.answer_to_image.overall_response',
    'answer_to_imagereason': 'vqac.answer_to_image.reason',
    'question_to_answerresponse': 'vqac.question_to_answer.response',
    'question_to_answeroverall_response': 'vqac.question_to_answer.overall_response',
    'question_to_answerreason': 'vqac.question_to_answer.reason',
    'guess_the_answerresponse': 'vqac.guess_the_answer.response',
    'guess_the_answerreason': 'vqac.guess_the_answer.reason',
    'reason_depthresponse': 'vqac.reason_depth.response',
    'reason_depthreason': 'vqac.reason_depth.reason',
}

raw_dataset.rename(columns=rename_mapping, inplace=True)

In [ ]:
list_columns = [
    'etp.txt_grammar.Reason_for_question',
    'etp.txt_grammar.Reason_for_answers',
    'etp.txt_unambiguity.Reason_for_question',
    'etp.txt_unambiguity.Reason_for_answers',
    'etp.txt_qa_structure.Reason_for_question',
    'etp.txt_qa_structure.Reason_for_answers',
    'etp.syntactic_complexity.Reason_for_question',
    'etp.syntactic_complexity.Reason_for_answers',
    'etp.language_naturalness.Reason_for_question',
    'etp.language_naturalness.Reason_for_answers',
    'eip.img_clarity.Reason',
    'eip.img_occlusion.Reason',
    'eip.img_diff_ability.Reason',
    'eip.img_interaction_level.Reason',
    'vqac.question_to_image.reason',
    'vqac.answer_to_image.reason',
    'vqac.question_to_answer.reason',
    'vqac.guess_the_answer.reason',
    'vqac.reason_depth.reason',
]

# Function to safely parse and convert to list
def to_list(x):
    try:
        if isinstance(x, str):
            x = ast.literal_eval(x)
        return sorted(x) if isinstance(x, list) else sorted([x])
    except Exception:
        return sorted([x])

# Apply the fix to each column if it exists
for col in list_columns:
    if col in raw_dataset.columns:
        raw_dataset[col] = raw_dataset[col].apply(to_list)

In [ ]:
processed_dataset = raw_dataset[rename_mapping.values()].copy(deep=True)

In [ ]:
display(processed_dataset)


print(pd.DataFrame({
    'non_null': processed_dataset.notnull().sum(),
    'dtype': processed_dataset.dtypes
}).sort_values('non_null'))

,eip.img_clarity.Score,eip.img_clarity.Reason,eip.img_occlusion.Score,eip.img_occlusion.Reason,eip.img_diff_ability.Score,eip.img_diff_ability.Reason,eip.img_object_density.Score,eip.img_interaction_level.Score,eip.img_interaction_level.Reason,eip.img_scene_clutter.Score,...,vqac.answer_to_image.response,vqac.answer_to_image.overall_response,vqac.answer_to_image.reason,vqac.question_to_answer.response,vqac.question_to_answer.overall_response,vqac.question_to_answer.reason,vqac.guess_the_answer.response,vqac.guess_the_answer.reason,vqac.reason_depth.response,vqac.reason_depth.reason
0,7,[The image has moderate clarity with some nois...,9,[The main subjects are fully visible without s...,9,"[Objects like the cake, candles, and people ar...",8,4,[There is a mild interaction where the woman a...,3,...,"['Yes', 'Yes', 'Yes', 'Yes', 'Yes']",Yes,"[The image shows two people., The image shows ...","['Yes', 'Yes', 'Yes', 'Yes', 'Yes']",Yes,"[The image shows two people., The image shows ...",No,[The question requires counting people based o...,3,[The question involves recognizing people in t...
1,7,[The image has moderate clarity with some nois...,8,"[The main subject, the traffic lights, are mos...",8,"[Objects like traffic lights, street signs, an...",6,2,[There are no clear interactions between objec...,3,...,"['Yes', 'Yes', 'Yes', 'No', 'Yes']",Yes,[The image does not show evening or night cond...,"['Yes', 'Yes', 'Yes', 'No', 'Yes']",Yes,[The image does not show evening or night cond...,Yes,[The question and answers are based on visual ...,2,[The question requires understanding the time ...
2,7,[The image has moderate clarity with some nois...,9,"[The main subject, the toilet, is fully visibl...",9,[The objects are clearly distinguishable by th...,5,2,[There are no clear interactions between the o...,3,...,"['No', 'No', 'No', 'No', 'No']",No,"[There is no phone in the image., There is no ...","['No', 'No', 'No', 'No', 'No']",No,[The question asks about the purpose of a phon...,No,[The question is about the purpose of a phone ...,1,[The question is straightforward and only requ...
3,7,[The image has moderate clarity with some nois...,8,"[The puppy is partially occluded by the shoe, ...",9,[The puppy and the shoe are easily distinguish...,4,2,[There is no clear interaction between the obj...,3,...,"['Yes', 'Yes', 'No', 'Yes', 'Yes']",Yes,"[The puppy appears to be resting., The puppy i...","['Yes', 'Yes', 'No', 'Yes', 'Yes']",Yes,"[The puppy appears to be resting., The puppy i...",No,[The question relies on visual content and spe...,3,[The question requires understanding the spati...
4,7,[The image has good clarity with clear details...,9,"[The main subject, the living room, is fully v...",9,"[Objects like furniture, appliances, and decor...",8,2,[There are no clear interactions between objec...,3,...,"['Yes', 'Yes', 'Yes', 'No', 'Yes']",Yes,"[The image shows a television, a ceiling fan, ...","['Yes', 'Yes', 'Yes', 'No', 'Yes']",Yes,"[The image shows a television, a ceiling fan, ...",Yes,[The question asks about furniture and objects...,2,[The question requires understanding the spati...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,7,[The image has good clarity with clear details...,9,"[The main subjects, the truck and the dog, are...",9,[The truck and the dog are easily distinguisha...,5,2,[There are no clear interactions between the o...,3,...,"['No', 'No', 'No', 'Yes', 'No']",No,[The dog appears to be standing near the truck...,"['No', 'No', 'No', 'Yes', 'No']",No,[The dog appears to be standing near the truck...,No,[The question requires specific actions observ...,2,[The question involves recognizing the dog's b...
5996,7,[The image has moderate clarity with some nois...,8,"[The main subject, the TV screen, is mostly vi...",8,"[Objects like the TV, cables, and the cat stat...",6,2,[There are no clear interactions between objec...,5,...,"['Yes', 'Yes', 'No', 'Yes', 'Yes']",Yes,"[The im

                                              non_null   dtype
idp.Image_mainobj_descrip                         3080  object
idp.Demographic_signals                           5972  object
idp.Cultural_context                              5999  object
eip.img_clarity.Score                             6000   int64
eip.img_diff_ability.Score                        6000   int64
eip.img_clarity.Reason                            6000  object
eip.img_occlusion.Score                           6000   int64
eip.img_occlusion.Reason                          6000  object
eip.img_interaction_level.Score                   6000   int64
eip.img_object_density.Score                      6000   int64
eip.img_diff_ability.Reason                       6000  object
eip.img_interaction_level.Reason                  6000  object
idp.Img_main_object                               6000  object
idp.Img_scene_type                                6000  object
eip.img_scene_clutter.Score                       6000 

In [ ]:
DIR_ROUGE_JSON = './data/quen-vl/res2.json'

from collections import defaultdict

def get_type_name(value):
    if isinstance(value, int):
        return 'int'
    elif isinstance(value, float):
        return 'float'
    elif isinstance(value, str):
        return 'str'
    elif isinstance(value, list):
        return 'list'
    elif isinstance(value, dict):
        return 'dict'
    elif value is None:
        return 'null'
    elif isinstance(value, bool):
        return 'bool'
    else:
        return type(value).__name__

def count_key_types(data, type_map=None, prefix=''):
    if type_map is None:
        type_map = defaultdict(set)

    if isinstance(data, dict):
        for key, value in data.items():
            full_key = f'{prefix}.{key}' if prefix else key
            type_map[full_key].add(get_type_name(value))
            count_key_types(value, type_map, full_key)
    elif isinstance(data, list):
        for item in data:
            count_key_types(item, type_map, prefix)

    return type_map

with open(DIR_ROUGE_JSON) as f:
    json_data = json.load(f)

print(len(json_data))

type_counts = count_key_types(json_data)

for key_path, types in type_counts.items():
    print(f'{key_path:<40} {len(types)} types:\\t{sorted(types)}')


6000
Img_scene_type                           1 types:	['str']
Img_main_object                          1 types:	['str']
rip                                      1 types:	['dict']
rip.Color                                1 types:	['str']
rip.Size                                 1 types:	['str']
rip.Shape                                1 types:	['str']
rip.Material                             1 types:	['str']
rip.Texture                              1 types:	['str']
rip.State or Action                      1 types:	['str']
Cultural_context                         1 types:	['str']
Demographic_signals                      1 types:	['str']
Scene_typicality_score                   1 types:	['int']
Image_mainobj_descrip                    2 types:	['dict', 'str']
Image_mainobj_descrip.Color              2 types:	['list', 'str']
Image_mainobj_descrip.Size               2 types:	['list', 'str']
Image_mainobj_descrip.Shape              2 types:	['list', 'str']
Image_mainobj_descrip.Material    

In [84]:
def extract_descrip(entry):
    if isinstance(entry, dict):
        if 'Image_mainobj_descrip' in entry:
            return entry['Image_mainobj_descrip']
        if 'rip' in entry:
            return entry['rip']
        for v in entry.values():
            result = extract_descrip(v)
            if result is not None:
                return result
    elif isinstance(entry, list):
        for item in entry:
            result = extract_descrip(item)
            if result is not None:
                return result
    return None

descriptions = [extract_descrip(entry) for entry in json_data]

print(len(descriptions))
print(descriptions[0])

6000
{'Color': 'various', 'Size': 'medium', 'Shape': 'human', 'Material': 'clothing', 'Texture': 'patterned', 'State or Action': 'sitting'}


In [85]:
def flatten_entry(x):
    if isinstance(x, dict):
        return ', '.join(f"{str(v).lower()} {k.lower()}" for k, v in x.items())
    elif isinstance(x, str):
        return x.lower()
    return ''

list_img_mainobj_desc = []

for item in descriptions:
    list_img_mainobj_desc.append(flatten_entry(item))


print(len(list_img_mainobj_desc))

6000


In [ ]:
processed_dataset['idp.Image_mainobj_descrip'] = list_img_mainobj_desc

info = pd.DataFrame({
    'non_null': processed_dataset.notnull().sum(),
    'dtype': processed_dataset.dtypes
}).sort_values('non_null')

print(info)

                                              non_null   dtype
idp.Demographic_signals                           5972  object
idp.Cultural_context                              5999  object
eip.img_clarity.Score                             6000   int64
eip.img_clarity.Reason                            6000  object
eip.img_diff_ability.Score                        6000   int64
eip.img_diff_ability.Reason                       6000  object
eip.img_occlusion.Score                           6000   int64
eip.img_occlusion.Reason                          6000  object
eip.img_interaction_level.Score                   6000   int64
eip.img_object_density.Score                      6000   int64
eip.img_interaction_level.Reason                  6000  object
eip.img_scene_clutter.Score                       6000   int64
idp.Img_main_object                               6000  object
idp.Img_scene_type                                6000  object
idp.Image_mainobj_descrip                         6000 

In [ ]:
for feature in ['idp.Demographic_signals', 'idp.Cultural_context']:
    processed_dataset.loc[:, feature] = processed_dataset[feature].fillna('none')


In [ ]:
info = pd.DataFrame({
    'non_null': processed_dataset.notnull().sum(),
    'dtype': processed_dataset.dtypes
}).sort_values('non_null')

print(info)

                                              non_null   dtype
eip.img_clarity.Score                             6000   int64
eip.img_clarity.Reason                            6000  object
eip.img_occlusion.Score                           6000   int64
eip.img_occlusion.Reason                          6000  object
eip.img_diff_ability.Score                        6000   int64
eip.img_diff_ability.Reason                       6000  object
eip.img_object_density.Score                      6000   int64
eip.img_interaction_level.Score                   6000   int64
eip.img_interaction_level.Reason                  6000  object
eip.img_scene_clutter.Score                       6000   int64
idp.Img_scene_type                                6000  object
idp.Img_main_object                               6000  object
idp.Image_mainobj_descrip                         6000  object
idp.Cultural_context                              6000  object
idp.Demographic_signals                           6000 

In [ ]:
processed_dataset.columns = processed_dataset.columns.str.replace('.', '_', regex=False)

In [ ]:
median_cols = [col for col in processed_dataset.columns if "Score_for_answers" in col]

for col in median_cols:
    new_col = f"{col}_median"
    processed_dataset.loc[:, new_col] = processed_dataset[col].apply(
        lambda x: np.median(ast.literal_eval(x)) if isinstance(x, str) and x.strip().startswith('[') else np.nan
    )

In [ ]:
value_cols = [
    "etp_txt_grammar_Score_for_question",
    "etp_txt_unambiguity_Score_for_question",
    "etp_txt_qa_structure_Score_for_question",
    "etp_syntactic_complexity_Score_for_question",
    "etp_language_naturalness_Score_for_question",
    "eip_img_clarity_Score",
    "eip_img_occlusion_Score",
    "eip_img_diff_ability_Score",
    "eip_img_object_density_Score",
    "eip_img_interaction_level_Score",
    "eip_img_scene_clutter_Score",
    "idp_Scene_typicality_score",
    "vqac_reason_depth_response",
    "etp_txt_grammar_Score_answers_median",
    "etp_txt_unambiguity_Score_answers_median",
    "etp_txt_qa_structure_Score_answers_median",
    "etp_syntactic_complexity_Score_answers_median",
    "etp_language_naturalness_Score_answers_median"
]

for col in value_cols:
    if col in processed_dataset.columns:
        processed_dataset[col] = pd.to_numeric(processed_dataset[col], errors='coerce')

        ovr_median = processed_dataset[col].median()
        label_col = f"{col}_Label"

        print(label_col, ovr_median)

        processed_dataset[label_col] = processed_dataset[col].apply(
            lambda x: "Passed" if pd.notna(x) and x >= ovr_median else "Failed"
        )


etp_txt_grammar_Score_for_question_Label 3.0
etp_txt_unambiguity_Score_for_question_Label 3.0
etp_txt_qa_structure_Score_for_question_Label 2.0
etp_syntactic_complexity_Score_for_question_Label 2.0
etp_language_naturalness_Score_for_question_Label 3.0
eip_img_clarity_Score_Label 7.0
eip_img_occlusion_Score_Label 8.0
eip_img_diff_ability_Score_Label 9.0
eip_img_object_density_Score_Label 6.0
eip_img_interaction_level_Score_Label 2.0
eip_img_scene_clutter_Score_Label 3.0
idp_Scene_typicality_score_Label 3.0
vqac_reason_depth_response_Label 2.0
